In [ ]:
!pip install shap
!pip install fasttreeshap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/crash_data_analysis(appliced sciences)"

/content/drive/MyDrive/Colab Notebooks/crash_data_analysis(appliced sciences)


In [ ]:
!ls
#查看文件

'balanced_remapping_data (1).csv'


In [ ]:
! pip install shap category_encoders imbalanced-learn seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 2.1 MB/s eta 0:00:00


In [ ]:
# !pip install numpy==1.26.4 --force-reinstall
!pip install numpy
!pip install shap --upgrade --quiet

In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")

# ========= 配置区 =========
FULL_SHAP = True          # True=对 X_test 全量计算; False=采样
SAMPLE_SIZE = 2000        # 采样数量 (FULL_SHAP=False 时生效)
APPROXIMATE = False       # True=近似更快；False=精确
CHECK_ADDITIVITY = True   # False 可提速但数值略变
BATCH_SIZE = 1000         # 批大小 (越大越快，占内存更多)
N_JOBS = -1               # 并行核数，-1=用尽可能多的核
# ========================

# 1) 读取数据
df = pd.read_csv("balanced_remapping_data (1).csv")

# 2) 定义特征与目标
categorical_features = ['lum', 'agg', 'int', 'atm', 'season', 'week', 'peak', 'catr',
                        'circ', 'vosp', 'prof', 'plan', 'surf', 'infra', 'situ', 'senc',
                        'catv', 'obs', 'obsm', 'manv', 'catu', 'sexe', 'trajet', 'secu']
numeric_features = ['nbv', 'occutc', 'age']
target = 'grav_balanced'
all_features = categorical_features + numeric_features

# 3) 删除缺失
df = df[all_features + [target]].dropna()

# 4) 类别变量转为 category
for col in categorical_features:
    df[col] = df[col].astype("category")

# 5) 训练集/测试集
X = df[all_features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6) 模型
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# -------------------------------
# 7) 待解释样本
# -------------------------------
if FULL_SHAP:
    X_eval = X_test.copy()
else:
    X_eval = shap.utils.sample(X_test, min(SAMPLE_SIZE, len(X_test)), random_state=42)

# 8) 背景数据 (用 kmeans 压缩加速)
try:
    X_bg = shap.kmeans(
        X_train.apply(lambda s: s.cat.codes if str(s.dtype) == 'category' else s),
        50, random_state=42
    )
except Exception:
    X_bg = X_train

# 9) 构建 explainer（interventional 避免报错）
explainer = shap.TreeExplainer(
    model,
    data=X_bg,
    feature_perturbation="interventional",  # ✅ 避免报错
    model_output="raw"
)

# 10) 分批并行 + tqdm
n = X_eval.shape[0]
n_batches = int(np.ceil(n / BATCH_SIZE))

def compute_batch(i):
    start = i * BATCH_SIZE
    end = min((i + 1) * BATCH_SIZE, n)
    return explainer.shap_values(
        X_eval.iloc[start:end],
        approximate=APPROXIMATE,
        check_additivity=CHECK_ADDITIVITY
    )

desc = f"Calculating SHAP ({'FULL' if FULL_SHAP else 'sampled'})"
shap_batches = Parallel(n_jobs=N_JOBS, prefer="threads")(
    delayed(compute_batch)(i) for i in tqdm(range(n_batches), desc=desc)
)

# 11) 拼接结果
if isinstance(shap_batches[0], list):  # 多分类
    n_classes = len(shap_batches[0])
    shap_values = [np.vstack([batch[c] for batch in shap_batches]) for c in range(n_classes)]
else:
    shap_values = np.vstack(shap_batches)

print("SHAP values structure:")
print(f"Type: {type(shap_values)}")
if isinstance(shap_values, list):
    print(f"Number of classes: {len(shap_values)}")
else:
    print(f"Shape: {getattr(shap_values, 'shape', None)}")
print(f"X_eval.shape = {X_eval.shape}")

# 12) 选择一个类别
if isinstance(shap_values, list):
    selected_shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
else:
    if len(shap_values.shape) == 3:
        selected_shap_values = shap_values[:, :, 1]
    else:
        selected_shap_values = shap_values

print(f"Selected SHAP values shape: {selected_shap_values.shape}")

# ====== A) dependence plot ======
X_eval_dep = X_eval.copy()
X_eval_dep['secu'] = X_eval_dep['secu'].astype(str)

plt.figure(figsize=(7,5))
shap.dependence_plot(
    'secu',
    selected_shap_values,
    X_eval_dep,
    interaction_index='age',
    show=False
)
# plt.title(f"SHAP Dependence Plot (secu & age) — {'FULL' if FULL_SHAP else 'sampled'}")
plt.title(f"SHAP Dependence Plot (secu & age), class_index = 1")
plt.tight_layout()
plt.savefig("!shap_dependence_secu_color_age.png", dpi=150)
plt.show()

# ====== B) age vs secu 散点图 ======
age_idx = list(X_eval.columns).index('age')
age_shap = selected_shap_values[:, age_idx]

X_scatter = X_eval.copy()
if str(X_scatter['secu'].dtype) == 'category':
    X_scatter['secu'] = X_scatter['secu'].cat.codes

x_val = X_scatter['secu'].astype(int).values
y_val = X_scatter['age'].values
c_val = age_shap

rng = np.random.default_rng(42)
x_jitter = x_val + rng.normal(0, 0.06, size=x_val.shape)

plt.figure(figsize=(7,5))
sc = plt.scatter(x_jitter, y_val, c=c_val, cmap='viridis', alpha=0.8)
cb = plt.colorbar(sc)
cb.set_label("SHAP value for age")
plt.xlabel("secu")
plt.ylabel("age")
plt.title(f"Age vs Secu (color = SHAP value for age) — {'FULL' if FULL_SHAP else 'sampled'}")
plt.xticks([0,1,2,3], ["无防护","安全带","儿童座椅","头盔"])
plt.tight_layout()
plt.show()


Calculating SHAP (FULL):   0%|          | 0/6 [00:00<?, ?it/s]

 16%|===                 | 467/3000 [06:59<37:52]       